# CineMatch — Top 500 Reviewers + Their Ratings

**Owner:** Geoff (CineMatch)  
**Goal:** Pull the top 500 popular reviewers and every (film, rating) pair from each. No review text — just the user-item-rating matrix.

## Outputs
- `data/users.csv` — username, name, location, total films rated
- `data/user_ratings.csv` — long format, **one row per (user, film, rating)**
- `cache_users/` — every fetched HTML page (re-runs are free)

## Time / disk
- ~500 users × ~30 ratings-pages avg × 0.6s ≈ **2.5 hours wall-clock**
- Cache: ~500 MB
- **Recommended: do a 25-user trial first** (set `MAX_USERS = 25`, ~7 min) to verify the pipeline, then bump to 500.


## §1. Install dependencies

In [1]:
%pip install --quiet requests beautifulsoup4 pandas tqdm


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## §2. Imports & config
**Set `MAX_USERS = 25` for your first run.** After §6 confirms data is flowing, change to `500` and re-run §2 + §7.

In [2]:
import json
import re
import time
import csv
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm

BASE_URL = "https://letterboxd.com"

# ---- Tune these -------------------------------------------------------------
MAX_USERS                  = 250
MAX_RATING_PAGES_PER_USER  = 50     # 50 pages × ~72 films/page = ~3600 most-recent rated films max
DELAY_SECONDS              = 0.6
ADAPTIVE_BACKOFF_BASE      = 30

REVIEWERS_URL = "https://letterboxd.com/reviewers/popular/this/all-time/"
REVIEWER_PAGES_TO_WALK = 20         # 20 × 25 = 500 users

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

OUTPUT_DIR    = Path("data");         OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR     = Path("cache_users");  CACHE_DIR.mkdir(exist_ok=True)
DEBUG_DIR     = Path("debug");        DEBUG_DIR.mkdir(exist_ok=True)
USERS_CSV     = OUTPUT_DIR / "users.csv"
RATINGS_CSV   = OUTPUT_DIR / "user_ratings.csv"
USERNAMES_CSV = OUTPUT_DIR / "usernames.csv"

session = requests.Session()
session.headers.update(HEADERS)

print(f"MAX_USERS                 = {MAX_USERS}")
print(f"MAX_RATING_PAGES_PER_USER = {MAX_RATING_PAGES_PER_USER}")
print(f"Outputs:                  {USERS_CSV}, {RATINGS_CSV}")


MAX_USERS                 = 250
MAX_RATING_PAGES_PER_USER = 50
Outputs:                  data/users.csv, data/user_ratings.csv


## §3. HTTP helper (cached + 429 backoff)

In [3]:
_consecutive_429 = 0


def cache_path_for(url):
    h = hashlib.md5(url.encode("utf-8")).hexdigest()
    return CACHE_DIR / f"{h}.html"


def fetch(url, retries=3, use_cache=True):
    global _consecutive_429
    cp = cache_path_for(url)
    if use_cache and cp.exists() and cp.stat().st_size > 0:
        return BeautifulSoup(cp.read_text(encoding="utf-8"), "html.parser")
    for attempt in range(retries):
        try:
            r = session.get(url, timeout=30)
            if r.status_code == 200:
                cp.write_text(r.text, encoding="utf-8")
                _consecutive_429 = 0
                time.sleep(DELAY_SECONDS)
                return BeautifulSoup(r.text, "html.parser")
            if r.status_code == 404:
                return None
            if r.status_code == 429:
                _consecutive_429 += 1
                wait = min(ADAPTIVE_BACKOFF_BASE * (2 ** (_consecutive_429 - 1)), 600)
                print(f"  [429] sleeping {wait}s")
                time.sleep(wait)
                continue
            print(f"  [{r.status_code}] {url}  (attempt {attempt + 1})")
        except requests.RequestException as exc:
            print(f"  [err] {url} -- {exc}")
        time.sleep(DELAY_SECONDS * (attempt + 2))
    return None


## §4. Collect top 500 reviewer usernames
Walks first 20 pages of `letterboxd.com/reviewers/popular/this/all-time/`. Tries 4 different selector strategies and reports which one worked.

In [4]:
RESERVED_USERNAMES = {
    "films", "lists", "members", "reviewers", "search", "about", "pro",
    "settings", "create-account", "sign-in", "sign-out", "year-in-review",
    "almanac", "journal", "contact", "terms", "privacy", "official",
    "showdown", "actor", "director", "studio", "country", "language",
    "genre", "theme", "mini-theme", "nanogenre", "decade", "year",
    "list", "tag", "story", "writer", "producer", "editor",
    "cinematography", "composer", "ajax", "api", "stats", "people",
    "crew", "cast", "fans", "likes", "reviews", "diary", "watchlist",
    "tv", "film", "rss", "feed", "popular", "new", "rated",
}


def extract_usernames_from_page(soup, verbose=False):
    """Try multiple strategies. Returns deduped list of usernames found on the page."""
    seen, out = set(), []

    def add(name):
        if not name or name in seen or name.lower() in RESERVED_USERNAMES:
            return
        seen.add(name)
        out.append(name)

    strategies = {}

    # A: legacy person-table
    a_count = len(out)
    for row in soup.select("table.person-table tbody tr"):
        a = row.select_one("h3.title-3 a[href]") or row.select_one("a[href^='/']")
        if a:
            href = a.get("href", "").strip("/")
            if href and "/" not in href:
                add(href)
    strategies["A (table.person-table)"] = len(out) - a_count

    # B: avatar links
    b_count = len(out)
    for a in soup.select("a.avatar[href^='/'], a[class*='avatar'][href^='/']"):
        href = a.get("href", "").strip("/")
        if href and "/" not in href:
            add(href)
    strategies["B (a.avatar)"] = len(out) - b_count

    # C: name links inside cards
    c_count = len(out)
    for a in soup.select("h3 a[href^='/'], h2 a[href^='/']"):
        href = a.get("href", "").strip("/")
        if href and "/" not in href:
            add(href)
    strategies["C (h3/h2 links)"] = len(out) - c_count

    # D: any single-segment anchor
    d_count = len(out)
    for a in soup.select("a[href]"):
        m = re.match(r"^/([A-Za-z0-9_-]+)/?$", a.get("href", ""))
        if m:
            add(m.group(1))
    strategies["D (any /<slug>/ anchor)"] = len(out) - d_count

    if verbose:
        print("  username strategies:")
        for k, v in strategies.items():
            print(f"    {k:35s} -> {v}")
        print(f"    TOTAL: {len(out)}")

    return out


def collect_top_reviewers(target=MAX_USERS, pages=REVIEWER_PAGES_TO_WALK):
    seen, ordered = set(), []
    pbar = tqdm(range(1, pages + 1), desc="Reviewer pages", unit="page")
    for pg in pbar:
        url = REVIEWERS_URL if pg == 1 else REVIEWERS_URL.rstrip("/") + f"/page/{pg}/"
        soup = fetch(url)
        if soup is None:
            break
        page_users = extract_usernames_from_page(soup, verbose=(pg == 1))
        new = 0
        for u in page_users:
            if u not in seen:
                seen.add(u); ordered.append(u); new += 1
        pbar.set_postfix(users=len(ordered), new=new)
        if new == 0:
            break
        if len(ordered) >= target:
            return ordered[:target]
    return ordered


top_users = collect_top_reviewers()
print(f"\nCollected {len(top_users)} top reviewers")

if len(top_users) == 0:
    print("\n⚠️  ZERO usernames collected. Skip ahead -- §6 has a hardcoded fallback test user.")
else:
    pd.DataFrame({"username": top_users, "rank": range(1, len(top_users) + 1)}).to_csv(USERNAMES_CSV, index=False)
    print(f"Saved to {USERNAMES_CSV}")
    print("Top 10:", top_users[:10])


Reviewer pages:   0%|          | 0/20 [00:00<?, ?page/s]

  username strategies:
    A (table.person-table)              -> 0
    B (a.avatar)                        -> 30
    C (h3/h2 links)                     -> 0
    D (any /<slug>/ anchor)             -> 4
    TOTAL: 34

Collected 250 top reviewers
Saved to data/usernames.csv
Top 10: ['kurstboy', 'schaffrillas', 'jay', 'deathproof', 'demiadejuyigbe', 'davidehrlich', 'thejoshl', 'ingridgoeswest', 'colonelmortimer', 'silentdawn']


## §5. Per-user ratings parser
The `/<username>/films/ratings/` page lists every film the user has rated, with the rating encoded as CSS class `rated-N` (N = 1..10, divide by 2 for star value).

In [5]:
def parse_profile(soup, username):
    name = username
    name_el = soup.select_one("h1.title-1, div.profile-name-wrap h1, h1.primaryname, h1")
    if name_el:
        candidate = name_el.get_text(strip=True)
        if candidate and not candidate.lower().startswith("letterboxd"):
            name = candidate
    location = None
    for span in soup.select("div.profile-metadata span.label, .metadatum span.label, .metadatum .label"):
        t = span.get_text(strip=True)
        if t and "http" not in t.lower():
            location = t
            break
    return {"username": username, "name": name, "location": location}


_RATED_RE = re.compile(r"rated-(\d+)")


def _find_rating_items(soup):
    """Find <li> containers that wrap a single (film, rating) pair."""
    # Strategy 1: known structure
    items = soup.select("ul.poster-list li, li.poster-container, li.griditem")
    items = [li for li in items if li.select_one("[data-film-slug], a[href^='/film/']")]
    if items:
        return items, "direct-selector"

    # Strategy 2: climb up from rating spans
    seen_ids, climbed = set(), []
    for el in soup.find_all(True):
        classes = el.get("class") or []
        if any(_RATED_RE.match(c) for c in classes):
            container = el.find_parent("li") or el.find_parent("article") or el.find_parent("div")
            if container is not None and id(container) not in seen_ids:
                seen_ids.add(id(container))
                climbed.append(container)
    if climbed:
        return climbed, "climb-from-rating"

    # Strategy 3: any <li> with a /film/ link
    fallback = [li for li in soup.find_all("li") if li.select_one("a[href^='/film/']")]
    return fallback, "li-with-film-link"


def parse_ratings_page(soup, verbose=False):
    """Return list of (slug, rating) pairs from a single ratings page."""
    items, strategy = _find_rating_items(soup)
    if verbose:
        print(f"   item strategy: {strategy} -> {len(items)} items")

    pairs = []
    for li in items:
        # Slug
        slug = None
        poster = li.select_one("[data-film-slug]")
        if poster:
            slug = poster.get("data-film-slug")
        if not slug:
            tl = li.select_one("[data-target-link]")
            if tl:
                m = re.match(r"/film/([^/]+)/?", tl.get("data-target-link", ""))
                if m:
                    slug = m.group(1)
        if not slug:
            a = li.select_one("a[href^='/film/']")
            if a:
                m = re.match(r"/film/([^/]+)/?", a.get("href", ""))
                if m:
                    slug = m.group(1)
        if not slug:
            continue

        # Rating
        rating = None
        for el in li.find_all(True):
            classes = el.get("class") or []
            for c in classes:
                m = _RATED_RE.match(c)
                if m:
                    try:
                        rating = int(m.group(1)) / 2.0
                    except ValueError:
                        pass
                    break
            if rating is not None:
                break

        if rating is not None:
            pairs.append((slug, rating))
    return pairs


def get_total_pages(soup):
    pagin = soup.select_one("div.paginate-pages")
    if not pagin:
        return 1
    nums = []
    for a in pagin.select("li a"):
        try:
            nums.append(int(a.get_text(strip=True)))
        except ValueError:
            continue
    return max(nums) if nums else 1


def scrape_user(username, max_pages=MAX_RATING_PAGES_PER_USER, verbose=False):
    """Returns ({profile dict}, [(slug, rating), ...])."""
    profile_soup = fetch(f"{BASE_URL}/{username}/")
    if profile_soup is None:
        return None, []
    profile = parse_profile(profile_soup, username)

    ratings_url = f"{BASE_URL}/{username}/films/ratings/"
    pg1 = fetch(ratings_url)
    if pg1 is None:
        return profile, []

    total = min(get_total_pages(pg1), max_pages)
    if verbose:
        print(f"   ratings pages (capped): {total}")

    pairs = parse_ratings_page(pg1, verbose=verbose)
    for pg in range(2, total + 1):
        soup = fetch(f"{ratings_url}page/{pg}/")
        if soup is None:
            break
        pairs.extend(parse_ratings_page(soup))
    return profile, pairs


## §6. Smoke test — fully diagnostic
Always prints HTTP status, Cloudflare detection, selector counts, parse output, and the first matched item's HTML. **If pairs come back 0, paste this entire output back to me.**

In [6]:
FALLBACK_TEST_USERS = ["davidehrlich", "lucy", "karsten", "demi", "filmsbyhanna"]

try:
    if top_users:
        test_user = top_users[0]
        test_source = "top_users[0]"
    else:
        test_user = FALLBACK_TEST_USERS[0]
        test_source = "fallback (top_users empty)"
except NameError:
    test_user = FALLBACK_TEST_USERS[0]
    test_source = "fallback (top_users not defined -- did you run §4?)"

print(f"Test user: {test_user}  ({test_source})")
print("=" * 70)

# ---- Stage 1: raw HTTP ------------------------------------------------------
import requests as _req
diag_url = f"{BASE_URL}/{test_user}/films/ratings/"
print(f"\nStage 1 -- raw fetch of: {diag_url}")
r = _req.get(diag_url, headers=HEADERS, timeout=30)
print(f"  HTTP status:    {r.status_code}")
print(f"  Final URL:      {r.url}")
print(f"  Content-Type:   {r.headers.get('Content-Type')}")
print(f"  Body length:    {len(r.text):,} chars")

body_lower = r.text[:5000].lower()
if "just a moment" in body_lower or "cf-browser-verification" in body_lower or "challenge-platform" in body_lower:
    print("  ⚠️  CLOUDFLARE CHALLENGE DETECTED -- Letterboxd is blocking the scraper.")
elif "letterboxd" not in body_lower:
    print("  ⚠️  Body doesn't mention 'letterboxd' -- this isn't the page we expected.")
else:
    print("  ✓ Looks like a real Letterboxd page")

(DEBUG_DIR / "smoke_ratings.html").write_text(r.text, encoding="utf-8")
print(f"  Saved HTML -> {(DEBUG_DIR / 'smoke_ratings.html').resolve()}")

# ---- Stage 2: selector counts -----------------------------------------------
print("\nStage 2 -- selector counts:")
soup = BeautifulSoup(r.text, "html.parser")
for sel in [
    "ul.poster-list li",
    "li.poster-container",
    "li.griditem",
    "[data-film-slug]",
    "[data-target-link]",
    "a[href^='/film/']",
    "[class*='rated-']",
]:
    print(f"  {sel:35s} -> {len(soup.select(sel))}")

# ---- Stage 3: run parser ----------------------------------------------------
print("\nStage 3 -- run parser:")
try:
    profile, pairs = scrape_user(test_user, verbose=True)
    print(f"\n  Profile: {profile}")
    print(f"  Ratings collected: {len(pairs)}")
except Exception as exc:
    print(f"  EXCEPTION: {exc!r}")
    import traceback; traceback.print_exc()
    profile, pairs = None, []

# ---- Stage 4: show output or first li ---------------------------------------
if pairs:
    print("\nStage 4 -- first 10 (slug, rating):")
    for slug, rating in pairs[:10]:
        print(f"  {slug:50s}  {rating}")
else:
    print("\nStage 4 -- 0 ratings parsed. First <li> on page:")
    first_li = soup.select_one("ul.poster-list li, [class*='rated-']")
    if first_li and first_li.name != "li":
        first_li = first_li.find_parent("li") or first_li
    if first_li:
        print(str(first_li)[:2000])
    else:
        print("  No <li> elements found. First 2000 chars of body:")
        print(r.text[:2000])

print("\n" + "=" * 70)
print("If 'Ratings collected' = 0: copy this entire output and paste back to Claude.")


Test user: kurstboy  (top_users[0])

Stage 1 -- raw fetch of: https://letterboxd.com/kurstboy/films/ratings/
  HTTP status:    403
  Final URL:      https://letterboxd.com/kurstboy/films/ratings/
  Content-Type:   text/html; charset=UTF-8
  Body length:    5,687 chars
  ⚠️  CLOUDFLARE CHALLENGE DETECTED -- Letterboxd is blocking the scraper.
  Saved HTML -> /Users/adonisgeoffmacias/Documents/GitHub/data-trio-project/DMW/NEW FOLDER/debug/smoke_ratings.html

Stage 2 -- selector counts:
  ul.poster-list li                   -> 0
  li.poster-container                 -> 0
  li.griditem                         -> 0
  [data-film-slug]                    -> 0
  [data-target-link]                  -> 0
  a[href^='/film/']                   -> 0
  [class*='rated-']                   -> 0

Stage 3 -- run parser:
   ratings pages (capped): 36
   item strategy: climb-from-rating -> 62 items

  Profile: {'username': 'kurstboy', 'name': 'kurstboy', 'location': 'los angeles'}
  Ratings collected: 224

## §7. Full run — all users, resume-safe
If kernel crashes, just re-run this cell. Already-scraped users are skipped.

In [7]:
def load_existing_progress():
    done = set()
    user_rows, rating_rows = [], []
    if USERS_CSV.exists():
        existing = pd.read_csv(USERS_CSV)
        done.update(existing["username"].astype(str).tolist())
        user_rows = existing.to_dict("records")
        print(f"Resuming: {len(done)} users already in {USERS_CSV.name}")
    if RATINGS_CSV.exists():
        existing = pd.read_csv(RATINGS_CSV)
        rating_rows = existing.to_dict("records")
        print(f"Resuming: {len(rating_rows)} ratings already in {RATINGS_CSV.name}")
    return done, user_rows, rating_rows


def write_outputs(user_rows, rating_rows):
    pd.DataFrame(user_rows).to_csv(USERS_CSV, index=False, quoting=csv.QUOTE_MINIMAL)
    pd.DataFrame(rating_rows, columns=["username", "film_slug", "rating"]).to_csv(
        RATINGS_CSV, index=False, quoting=csv.QUOTE_MINIMAL
    )


# ---- run ----
done, user_rows, rating_rows = load_existing_progress()
target = top_users[:MAX_USERS]
todo = [u for u in target if u not in done]
print(f"\nTarget: {len(target)}. Done: {len(done)}. To scrape now: {len(todo)}\n")

pbar = tqdm(todo, desc="Scraping users", unit="user")
for i, username in enumerate(pbar, start=1):
    profile, pairs = scrape_user(username)
    if profile is None:
        pbar.set_postfix(user=username, status="skip")
        continue
    profile["total_films_rated"] = len(pairs)
    profile["date_scraped"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    user_rows.append(profile)
    for slug, rating in pairs:
        rating_rows.append({"username": username, "film_slug": slug, "rating": rating})

    pbar.set_postfix(user=username[:18], ratings=len(pairs), total=len(rating_rows))

    if i % 10 == 0:
        write_outputs(user_rows, rating_rows)

write_outputs(user_rows, rating_rows)
print(f"\nDone. {len(user_rows)} users / {len(rating_rows)} ratings -> {OUTPUT_DIR}/")



Target: 250. Done: 0. To scrape now: 250



Scraping users:   0%|          | 0/250 [00:00<?, ?user/s]

  [403] https://letterboxd.com/usercillian/films/ratings/page/13/  (attempt 1)
  [403] https://letterboxd.com/usercillian/films/ratings/page/13/  (attempt 2)
  [403] https://letterboxd.com/usercillian/films/ratings/page/13/  (attempt 3)
  [403] https://letterboxd.com/sophiedarcy/films/ratings/  (attempt 1)
  [403] https://letterboxd.com/gift-guide/films/ratings/  (attempt 1)
  [403] https://letterboxd.com/gift-guide/films/ratings/  (attempt 2)
  [403] https://letterboxd.com/gift-guide/films/ratings/  (attempt 3)
  [403] https://letterboxd.com/willhunting/films/ratings/page/16/  (attempt 1)
  [403] https://letterboxd.com/willhunting/films/ratings/page/16/  (attempt 2)
  [403] https://letterboxd.com/willhunting/films/ratings/page/16/  (attempt 3)
  [403] https://letterboxd.com/barbiesswanlake/films/ratings/  (attempt 1)
  [403] https://letterboxd.com/barbiesswanlake/films/ratings/  (attempt 2)
  [403] https://letterboxd.com/barbiesswanlake/films/ratings/  (attempt 3)
  [403] https://lett

## §8. Inspect

In [8]:
users_df = pd.read_csv(USERS_CSV)
ratings_df = pd.read_csv(RATINGS_CSV)

print(f"users.csv:        {users_df.shape}")
print(f"user_ratings.csv: {ratings_df.shape}")
print(f"\nUnique users:     {ratings_df['username'].nunique()}")
print(f"Unique films:     {ratings_df['film_slug'].nunique()}")

print("\nRatings per user (top 10):")
print(ratings_df.groupby("username").size().sort_values(ascending=False).head(10))

print("\nMost-rated films (top 20):")
print(ratings_df.groupby("film_slug").size().sort_values(ascending=False).head(20))

print("\nRating value distribution:")
print(ratings_df["rating"].value_counts().sort_index())

ratings_df.head(10)


users.csv:        (250, 5)
user_ratings.csv: (502749, 3)

Unique users:     241
Unique films:     58136

Ratings per user (top 10):
username
bertz             3600
elcochran90       3600
fuchsiadyke       3600
mcumagik          3599
gemko             3599
jbird26           3599
dirkh             3597
theriverjordan    3597
cinemavoid        3596
russman           3595
dtype: int64

Most-rated films (top 20):
film_slug
parasite-2019                       221
once-upon-a-time-in-hollywood       218
knives-out-2019                     217
get-out-2017                        213
la-la-land                          212
nope                                212
midsommar                           209
lady-bird                           208
barbie                              208
dune-2021                           207
arrival-2016                        207
sinners-2025                        206
challengers                         206
spider-man-into-the-spider-verse    206
uncut-gems        

,username,film_slug,rating
0,kurstboy,michael-2026,2.0
1,kurstboy,mother-mary-2026,3.5
2,kurstboy,faces-of-death-2026,4.0
3,kurstboy,the-super-mario-galaxy-movie,1.5
4,kurstboy,the-drama,4.0
5,kurstboy,pizza-movie-2026,3.0
6,kurstboy,project-hail-mary,3.5
7,kurstboy,the-bride-2026,2.5
8,kurstboy,hoppers,4.0
9,kurstboy,crime-101,3.0


## §9. (Optional) Pivot into wide user-item matrix
For recommender models that want a dense matrix.

In [9]:
top_films = ratings_df.groupby("film_slug").size().nlargest(500).index
matrix = (ratings_df[ratings_df["film_slug"].isin(top_films)]
           .pivot_table(index="username", columns="film_slug", values="rating"))
print(f"Matrix shape (top 500 films): {matrix.shape}")
print(f"Density: {matrix.notna().sum().sum() / matrix.size:.2%}")
matrix.iloc[:5, :8]


Matrix shape (top 500 films): (240, 500)
Density: 65.06%


film_slug,10-cloverfield-lane,10-things-i-hate-about-you,12-years-a-slave,1917,20th-century-women,21-jump-street,28-days-later,28-years-later
username,,,,,,,,
812filmreviews,2.5,NaN,5.0,3.0,3.5,4.0,4.0,4.5
aarnwlsn,3.0,5.0,3.0,NaN,4.5,5.0,3.5,4.5
aksually,4.0,3.5,NaN,4.0,NaN,NaN,NaN,NaN
alexcolemann,3.5,4.0,NaN,NaN,NaN,5.0,4.0,4.5
alexlawther,4.0,5.0,NaN,NaN,4.0,NaN,3.0,NaN


## §10. Fetch TMDB IDs and Titles
We only want to scrape each movie ONCE, so we'll get the unique slugs from our ratings data.

In [ ]:

print("Loading ratings data to find unique films...")
final_ratings_df = pd.read_csv(RATINGS_CSV)
unique_slugs = final_ratings_df['film_slug'].dropna().unique()

print(f"Found {len(unique_slugs)} unique films. Starting fetch...")

film_details = []
pbar = tqdm(unique_slugs, desc="Fetching TMDB IDs", unit="film")

for slug in pbar:
    url = f"{BASE_URL}/film/{slug}/"
    soup = fetch(url, use_cache=True) # Reusing your awesome fetch() function!
    
    if not soup:
        continue
        
    # 1. Get the Movie Title
    title = slug # Fallback
    title_tag = soup.select_one("meta[property='og:title']")
    if title_tag and title_tag.get("content"):
        # This usually returns "The Devil Wears Prada (2006)"
        # We can split by ' (' to remove the year if we just want the title
        title = title_tag.get("content").split(' (')[0] 
    
    # 2. Get the TMDB ID
    tmdb_id = None
    # Find any link pointing to TMDB movies
    tmdb_link = soup.select_one("a[href*='themoviedb.org/movie/']")
    if tmdb_link:
        href = tmdb_link.get("href", "")
        # Regex to pull out the digits right after /movie/
        match = re.search(r'/movie/(\d+)', href)
        if match:
            tmdb_id = match.group(1)
            
    film_details.append({
        "film_slug": slug,
        "movie_title": title,
        "movie_id": tmdb_id
    })
    
    pbar.set_postfix(title=title[:15], tmdb=tmdb_id)

# 3. Create a DataFrame and merge it back with our user ratings
films_df = pd.DataFrame(film_details)

print("\nMerging data...")
# Merge on 'film_slug' to bring title and ID into the main dataset
final_df = final_ratings_df.merge(films_df, on="film_slug", how="left")

# Reorder columns to exactly what you requested: user, movie title, rating, movie id
final_df = final_df[["username", "movie_title", "rating", "movie_id"]]

FINAL_CSV = OUTPUT_DIR / "final_tmdb_ratings.csv"
final_df.to_csv(FINAL_CSV, index=False)

print(f"Success! Final dataset saved to {FINAL_CSV}")
final_df.head(10)

Loading ratings data to find unique films...
Found 58136 unique films. Starting fetch...


Fetching TMDB IDs:   0%|          | 0/58136 [00:00<?, ?film/s]